# Mathematical Framework for Comparing Population Simulators

**Monte Carlo, Transition Matrices, and Extensions to Markov Models**

Econ-ARK project, 2026

---

## 1. Introduction and Motivation

This document provides a unified mathematical framework for understanding how population distributions are computed in heterogeneous-agent models after the individual optimization problem has been solved.

The two main computational approaches are:

1. **Monte Carlo (MC) simulation**: track $N$ individual agents through stochastic transitions, approximating the population distribution with an empirical measure.

2. **Transition matrix (TM) methods**: discretize the state space onto a grid, build a Markov transition matrix from the solved policy function, and propagate a probability vector deterministically.

These methods exhibit a fundamental **bias–variance tradeoff**: MC is unbiased but noisy; TM is deterministic but introduces grid discretization error. Understanding this tradeoff—and how it varies with model complexity—is the central theme.

The framework covers:

- The basic single-state IndShock model (Sections 2–9)
- Extension to **Markov-switching models** with discrete aggregate states (Section 10)
- The **Harmenberg neutral measure** for collapsing the permanent income dimension (Section 11)
- **General equilibrium** (Krusell–Smith) with endogenous prices (Section 12)
- **Sequence-space Jacobians** for impulse response computation (Section 13)

The notation follows the “perch” structure from the Bellman-DDSL framework, which decomposes each period into arrival, decision, and continuation states connected by transition functions. This decomposition clarifies exactly where MC draws shocks (creating variance) and where TM applies the lottery method (creating bias).

For a broader survey of simulation methods in heterogeneous-agent macroeconomics, see Algan et al. (2014); for the benchmark comparison, see den Haan (2010); for the non-stochastic simulation method, see Young (2010).

## 2. The Problem Structure

We study models in which a continuum of agents solve individual optimization problems and a population distribution evolves as a consequence of their optimal decisions and stochastic shocks.  The full problem decomposes into three layers:

1. **Individual optimization** (the Bellman equation): produces policy functions at each decision perch.
2. **Distribution evolution** (the Markov operator): given the policy functions, propagates the cross-sectional distribution of agents forward through the perch structure.
3. **Aggregation**: computes population-level statistics from the distribution.

Monte Carlo (MC) and transition matrix (TM) methods differ in how they carry out layers 2 and 3.  Layer 1 is shared: both methods take the same solved policy functions as input.

### Hierarchical structure

The individual problem has a hierarchical organization:

- A **period** $\mathbb{S}$ consists of an ordered sequence of **stages**: $\mathbb{S}[0], \mathbb{S}[1], \ldots, \mathbb{S}[-1]$.
- Each **stage** contains three **perches**: arrival ($a$), decision ($v$), and continuation ($e$).
- Within a stage, **transition functions** connect perches: $g_{av}$ (arrival $\to$ decision) and $g_{ve}$ (decision $\to$ continuation).
- Between stages, **connector functions** link continuation to the next arrival: $g_{ea+}$ (sequential) or $g_{va+}$ (branching).

This hierarchy is the key organizational principle from the Bellman-DDSL framework.  For the simulation comparison, the critical insight is that the one-period forward operator $\mathcal{T}^*$ on the population distribution decomposes into a sequence of measure transitions through these perches.

## 3. Stage Structure and Notation

### 3.1 The three perches

Each stage is defined by three perches, each with an associated state space and value function:

| Perch | State space | State variable | Value function | Description |
|-------|-------------|---------------|----------------|-------------|
| **Arrival** | $\mathcal{X}_a$ | $x_a$ | $\mathcal{A}(x_a)$ | Agent's state before shocks realize |
| **Decision** | $\mathcal{X}_v$ | $x_v$ | $\mathcal{V}(x_v)$ | Agent's state after shocks, before choice |
| **Continuation** | $\mathcal{X}_e$ | $x_e$ | $\mathcal{E}(x_e)$ | Agent's state after choice, before next stage |

### 3.2 Within-stage transitions

Two transition functions connect the perches within a stage:

$$
g_{av} : \mathcal{X}_a \times \mathcal{Z}_{av} \to \mathcal{X}_v, \qquad x_v = g_{av}(x_a, \zeta_{av})
$$

$$
g_{ve} : \mathcal{X}_v \times \Pi \to \mathcal{X}_e, \qquad x_e = g_{ve}(x_v, \pi)
$$

where $\mathcal{Z}_{av}$ is the pre-decision shock space and $\Pi(x_v)$ is the feasible choice set.

### 3.3 Between-stage connectors

For sequential stages, a connector function maps the continuation state to the next stage's arrival:

$$
g_{ea+} : \mathcal{X}_e \to \mathcal{X}_{a+}, \qquad x_{a+} = g_{ea+}(x_e).
$$

### 3.4 The Bellman equation in perch notation

$$
\mathcal{V}(x_v) = \max_{\pi \in \Pi(x_v)} \bigl[ r(x_v, \pi) + \beta(x_v) \, \mathcal{E}\bigl(g_{ve}(x_v, \pi)\bigr) \bigr]
$$

$$
\mathcal{A}(x_a) = \mathbb{E}_{\zeta_{av}}\bigl[\mathcal{V}\bigl(g_{av}(x_a, \zeta_{av})\bigr)\bigr]
$$

The optimal policy function is $\pi^*(x_v) = \arg\max_{\pi \in \Pi(x_v)}[\cdots]$.

### 3.5 Notation summary

#### Spaces and measures

| Symbol | Meaning |
|--------|--------|
| $\mathcal{X}_a, \mathcal{X}_v, \mathcal{X}_e$ | Arrival, decision, and continuation state spaces |
| $\mathcal{Z}_{av}$ | Pre-decision shock space |
| $\Pi(x_v) \subseteq \Pi$ | Feasible choice set at decision state $x_v$ |
| $\mu_a, \mu_v, \mu_e$ | Population measures over arrival, decision, continuation states |

#### Functions and operators

| Symbol | Meaning |
|--------|--------|
| $\mathcal{A}(x_a), \mathcal{V}(x_v), \mathcal{E}(x_e)$ | Value functions at each perch |
| $\pi^*(x_v)$ | Optimal policy at the decision perch |
| $g_{av}, g_{ve}, g_{ea+}$ | Transition and connector functions |
| $\mathcal{T}^*$ | One-period forward operator on measures: $\mu_{a+} = \mathcal{T}^* \mu_a$ |
| $\mu_a^*$ | Invariant distribution: $\mu_a^* = \mathcal{T}^* \mu_a^*$ |

#### Aggregate quantities

For any integrable function $h : \mathcal{X}_v \to \mathbb{R}$, the population aggregate at the decision perch is

$$
\bar{h}_t = \int_{\mathcal{X}_v} h(x_v) \, d\mu_{v,t}(x_v).
$$

## 4. Rosetta Stone: Mapping to HARK Models

### 4.1 IndShockConsumerType

| Abstract | Concrete | Description |
|----------|----------|-------------|
| $\mathcal{X}_a$ | $\mathbb{R}_{++}$ | Arrival state space (bank balances) |
| $x_a$ | $b$ | Bank balance at start of period |
| $\mathcal{X}_v$ | $\mathbb{R}_{++}$ | Decision state space (market resources) |
| $x_v$ | $m$ | Market resources after income realization |
| $\mathcal{X}_e$ | $\mathbb{R}_+$ | Continuation state space (end-of-period assets) |
| $x_e$ | $a$ | End-of-period assets |
| $\Pi(x_v)$ | $(0, m)$ | Choice set: consume between 0 and $m$ |
| $\pi$ | $c$ | Choice variable: consumption |
| $\mathcal{Z}_{av}$ | $\{(\psi, \theta)\}$ | Permanent and transitory income shocks |
| $g_{av}(b, \zeta)$ | $m = \frac{R}{\Gamma \psi} b + \theta$ | Arrival $\to$ decision |
| $g_{ve}(m, c)$ | $a = m - c$ | Decision $\to$ continuation |
| $g_{ea+}(a)$ | $b_+ = a$ | Connector: assets become next bank balance |
| $r(m, c)$ | $u(c) = \frac{c^{1-\rho}}{1-\rho}$ | CRRA utility |

**Value function equations:**

$$
\mathcal{V}(m) = \max_{c \in (0, m)} \left[ u(c) + \beta \, \mathcal{E}(m - c) \right], \qquad
\mathcal{A}(b) = \mathbb{E}_{(\psi, \theta)} \left[ \mathcal{V}\!\left(\frac{R}{\Gamma \psi} b + \theta\right) \right]
$$

### 4.2 MarkovConsumerType

When the agent faces a discrete Markov state $j \in \{0, \ldots, J-1\}$, the state space augments to $(m, j)$ and the transition includes the Markov transition:

| Abstract | Concrete | Description |
|----------|----------|-------------|
| $\mathcal{X}_v$ | $\mathbb{R}_{++} \times \{0,\ldots,J{-}1\}$ | Decision state: $(m, j)$ |
| $\mathcal{Z}_{av}$ | $\{(\psi, \theta, j')\}$ | Income shocks + Markov transition |
| $g_{av}((b,j), (\psi,\theta,j'))$ | $m = \frac{R_{j'}}{\Gamma_{j'} \psi} b + \theta$ | Arrival $\to$ decision in new state $j'$ |
| $\pi^*_j(m)$ | $c^*_j(m)$ | State-dependent consumption function |
| $\Pr(j' \mid j)$ | $\texttt{MrkvArray}[j, j']$ | Row-stochastic Markov matrix |

The Markov transition and idiosyncratic shocks are independent: $\Pr(j', \psi, \theta \mid j) = \texttt{MrkvArray}[j,j'] \cdot Q(\psi, \theta)$.

### 4.3 HARK API mapping

| HARK method | Mathematical operation |
|-------------|----------------------|
| `agent.solve()` | Solve Bellman: $\mathcal{V}(x_v) = \max_\pi [r + \beta \, \mathcal{E}(g_{ve})]$ |
| `agent.simulate()` | MC: per-agent traversal of $\Gamma_{av} \to \Gamma_{ve} \to \Gamma_{ea+}$ |
| `define_distribution_grid()` | Discretize $\mathcal{X}_v$ into grid $\mathcal{G}$ |
| `calc_transition_matrix()` | Build $\boldsymbol{\Pi} \approx \text{discretize}(\Gamma_{ea+} \circ \Gamma_{ve} \circ \Gamma_{av})$ |
| `calc_ergodic_dist()` | Find $\mathbf{p}^* = \boldsymbol{\Pi} \mathbf{p}^*$ via eigenvector |
| `compute_pe_steady_state()` | Solve + TM + ergodic dist + aggregate $C, A$ |
| `calc_jacobian(shk, T)` | SSJ Jacobians via Fake News Algorithm |
| `neutral_measure = True` | Harmenberg: collapse $\mathcal{X}_a$ from 2D to 1D |
| `jump_to_grid_1D/2D` | Lottery method (mean-preserving grid projection) |
| `gen_tran_matrix_1D_markov` | Block-structured $\boldsymbol{\Pi}$ for Markov models |

## 5. The Exact Distribution Operator

The one-period forward operator $\mathcal{T}^*$ on population measures decomposes into a sequence of perch-level measure transitions.

### 5.1 Within-stage measure transitions

**Arrival $\to$ Decision** ($\Gamma_{av}$): Shocks realize, mixing the arrival distribution with the shock distribution.

$$
\mu_v(B) = \int_{\mathcal{X}_a} Q\bigl(\{\zeta : g_{av}(x_a, \zeta) \in B\}\bigr) \, d\mu_a(x_a)
$$

This is where **stochastic mixing** occurs: each arrival state fans out into multiple decision states according to the shock distribution $Q$.

**Decision $\to$ Continuation** ($\Gamma_{ve}$): The policy function maps each decision state deterministically.

$$
\mu_e(C) = \mu_v\!\left(\{x_v : g_{ve}(x_v, \pi^*(x_v)) \in C\}\right)
$$

This is a **deterministic pushforward** (given the solved policy).

### 5.2 Between-stage connector transition

$$
\mu_{a+}(A) = \mu_e\!\left(\{x_e : g_{ea+}(x_e) \in A\}\right)
$$

When $g_{ea+}$ is the identity (as in the single-stage model where $b_+ = a$), this is simply $\mu_{a+} = \mu_e$.

### 5.3 The composite one-period operator

$$
\mathcal{T}^* = \Gamma_{ea+} \circ \Gamma_{ve} \circ \Gamma_{av}
$$

That is, $\mu_{a,t+1} = \mathcal{T}^* \mu_{a,t}$.

### 5.4 Adjoint structure

The **forward operator** $\mathcal{T}^*$ (Kolmogorov Forward) pushes measures forward. Its adjoint $\mathcal{T}$ (Kolmogorov Backward) acts on functions:

$$
({\mathcal{T}} f)(x_a) = \mathbb{E}_\zeta\!\left[ f\!\left( g_{ea+}\!\left(g_{ve}\!\left(g_{av}(x_a, \zeta),\, \pi^*(g_{av}(x_a, \zeta))\right)\right)\right) \right]
$$

The duality relation $\int f \, d(\mathcal{T}^* \mu) = \int (\mathcal{T} f) \, d\mu$ connects the HJB equation (backward) to the KF equation (forward) as transposes (Achdou et al., 2022).

### 5.5 Steady-state distribution

The ergodic distribution $\mu_a^*$ satisfies $\mu_a^* = \mathcal{T}^* \mu_a^*$. Under standard conditions (Feller property, compactness, irreducibility, aperiodicity), existence and uniqueness are guaranteed (Santos and Peralta-Alva, 2005).

### 5.6 Transition dynamics

Starting from $\mu_{a,0}$, the path is $\mu_{a,t} = (\mathcal{T}^*)^t \mu_{a,0}$. This is exact but requires working with the infinite-dimensional object $\mu_{a,t}$. The two simulation methods approximate this path in fundamentally different ways.

## 6. Method 1: Monte Carlo Simulation

### 6.1 The approximation in perch notation

MC replaces the arrival measure $\mu_{a,t}$ with an **empirical measure** supported on $N$ agent states:

$$
\hat{\mu}_{a,t}^N = \frac{1}{N} \sum_{i=1}^{N} \delta_{x_{a,t}^{(i)}}
$$

Each agent traverses the perch structure independently each period:

**Step 1 (Arrival $\to$ Decision):** Draw shock and compute decision state.

$$
\zeta_{av}^{(i)} \sim Q, \qquad x_{v,t}^{(i)} = g_{av}\!\left(x_{a,t}^{(i)},\, \zeta_{av}^{(i)}\right)
$$

**Step 2 (Decision $\to$ Continuation):** Apply the solved policy function.

$$
\pi^{(i)} = \pi^*\!\left(x_{v,t}^{(i)}\right), \qquad x_{e,t}^{(i)} = g_{ve}\!\left(x_{v,t}^{(i)},\, \pi^{(i)}\right)
$$

**Step 3 (Connector):** Map to next period's arrival.

$$
x_{a,t+1}^{(i)} = g_{ea+}\!\left(x_{e,t}^{(i)}\right)
$$

### 6.2 Implementation note: newborn transitory shocks in HARK

> **Note.** HARK's `get_shocks()` method suppresses transitory income shocks for agents with `t_age = 0` (when `NewbornTransShk = False`, the default), forcing $\theta = 1.0$ in the first period after birth or rebirth. This means newborn agents always receive the mean transitory shock, which biases first-period consumption upward in MC simulations with mortality (where agents are continually reborn).
>
> The TM method is unaffected because it constructs transitions using the full shock distribution at every grid point.
>
> **Workaround:** After calling `agent.initialize_sim()`, set `agent.t_age = np.ones(AgentCount, dtype=int)` so that all agents are treated as age-1 (past the suppression) from the start. This ensures the full transitory shock distribution is applied from the first simulated period.

### 6.3 Aggregate computation

$$
\hat{h}_t^N = \frac{1}{N} \sum_{i=1}^N h\!\left(x_{v,t}^{(i)}\right)
$$

### 6.4 Error structure

**Sampling error (variance).** By the CLT, for fixed $t$:

$$
\sqrt{N}\bigl(\hat{h}_t^N - \bar{h}_t\bigr) \xrightarrow{d} \mathcal{N}\bigl(0, \operatorname{Var}_{\mu_{v,t}}[h]\bigr).
$$

**No discretization bias.** Agents live in the continuous state spaces. The MC estimate is unbiased: $\mathbb{E}[\hat{h}_t^N] = \bar{h}_t$.

### 6.5 Key properties

| Property | MC characteristic |
|----------|------------------|
| State spaces | Continuous $\mathcal{X}_a, \mathcal{X}_v, \mathcal{X}_e$ (no grids) |
| Bias | Zero (unbiased for any $N$) |
| Variance | $O(1/N)$ per aggregate |
| Aggregate time series | Fluctuates (sampling noise from shock draws at $\Gamma_{av}$) |
| Steady-state aggregates | Noisy; require large $N$ and long burn-in |
| Individual histories | Available (full trajectories through all perches) |

## 7. Method 2: Transition Matrix Simulation

### 7.1 State space discretization

The TM method replaces the continuous arrival state space $\mathcal{X}_a$ with a finite grid $\mathcal{G}_a = \{g_1, \ldots, g_M\}$ of $M$ points. The distribution is a probability vector:

$$
\mathbf{p}_{a,t} \in \mathbb{R}^M, \qquad p_{a,t,j} \geq 0, \qquad \sum_{j=1}^M p_{a,t,j} = 1.
$$

### 7.2 The lottery method at perch transitions

For each grid point $g_j$ and each discretized shock $\zeta_k$ with probability $q_k$:

1. Compute $x_v = g_{av}(g_j, \zeta_k)$
2. Apply the policy: $x_e = g_{ve}(x_v, \pi^*(x_v))$
3. Compute next arrival: $x_{a+} = g_{ea+}(x_e)$

The resulting $x_{a+}$ generically falls between grid points $g_i$ and $g_{i+1}$. The lottery assigns:

$$
\omega = \frac{x_{a+} - g_i}{g_{i+1} - g_i}, \qquad \text{fraction } (1-\omega) \text{ to } g_i, \quad \text{fraction } \omega \text{ to } g_{i+1}.
$$

This preserves the conditional mean: $(1-\omega) g_i + \omega \, g_{i+1} = x_{a+}$.

### 7.3 Constructing the transition matrix

$$
\Pi_{ij} = \sum_k q_k \cdot w_{ijk},
$$

where $w_{ijk}$ is the lottery weight from grid point $j$ to grid point $i$ under shock $\zeta_k$. The matrix is column-stochastic ($\sum_i \Pi_{ij} = 1$ for all $j$), so that $\mathbf{p}_{a,t+1} = \boldsymbol{\Pi} \, \mathbf{p}_{a,t}$.

### 7.4 Ergodic distribution

$$
\mathbf{p}_a^* = \boldsymbol{\Pi} \, \mathbf{p}_a^*, \qquad \sum_j p_{a,j}^* = 1.
$$

Solved in HARK's `calc_ergodic_dist()` using `scipy.sparse.linalg.eigs`.

### 7.5 Aggregate computation

$$
\tilde{h}_t = \mathbf{h}^\top \mathbf{p}_{v,t} = \sum_{j=1}^M h(g_{v,j}) \, p_{v,t,j}.
$$

There is no sampling noise.

### 7.6 Error structure

Three sources of bias:

1. **Grid resolution:** Coarse grids fail to capture fine distributional structure, especially in the tails.

2. **Lottery error:** The jump-to-grid allocation preserves the conditional mean but underestimates the conditional variance:

$$
\operatorname{Var}_{\text{lottery}}[x_{a+} \mid x_a, \zeta] = \omega(1-\omega)(g_{i+1} - g_i)^2 < \operatorname{Var}_{\text{true}}[x_{a+} \mid x_a, \zeta].
$$

3. **Tail truncation:** The grid has finite bounds. Mass beyond `mMax` is forced to the boundary.

**Zero variance:** Given $\boldsymbol{\Pi}$ and $\mathbf{p}_{a,0}$, the entire path is deterministic.

### 7.7 Key properties

| Property | TM characteristic |
|----------|------------------|
| State spaces | Finite grid $\mathcal{G} \subset \mathcal{X}_a$ with $M$ points |
| Bias | Non-zero (discretization at each perch transition) |
| Variance | Zero (deterministic) |
| Aggregate time series | Constant at steady state (flat line) |
| Steady-state aggregates | Exact for the discretized model |
| Individual histories | Not available |

## 8. Comparing the Two Approximations

### 8.1 The precision–accuracy tradeoff

$$
\text{MSE} = \text{Bias}^2 + \text{Variance}.
$$

| | MC | TM |
|--|----|----|  
| **Bias** | 0 | $O(\Delta g)$, decreasing in grid fineness |
| **Variance** | $O(1/N)$, decreasing in agent count | 0 |

MC is **accurate** (unbiased) but **imprecise** (noisy). TM is **precise** (deterministic) but **less accurate** (discretization error).

### 8.2 Where the errors enter in the perch structure

| Perch transition | MC error source | TM error source |
|-----------------|----------------|-----------------|
| $\Gamma_{av}$ (arrival $\to$ decision) | Shock sampling variance ($N$ iid draws) | Shock discretization (finite $\zeta_k$) |
| $\Gamma_{ve}$ (decision $\to$ continuation) | None (deterministic given $\pi^*$) | Policy evaluation on grid only |
| $\Gamma_{ea+}$ (connector) | None (exact mapping) | Lottery projection onto grid |

### 8.3 Computational costs

| Operation | MC cost | TM cost |
|-----------|---------|---------|  
| One-period forward | $O(N)$ per period | $O(M^2)$ to build $\boldsymbol{\Pi}$; $O(M)$ per multiply |
| Steady-state distribution | Long simulation ($N \times T$ draws) | Eigenvalue problem ($O(M^2)$ sparse) |
| Steady-state aggregates | Sample means from history | Inner product $\mathbf{h}^\top \mathbf{p}^*$ |
| Transition dynamics | Re-simulate $N$ agents each period | Matrix-vector multiply per period |
| Jacobians (SSJ) | Not directly available | Required input (Auclert et al., 2021) |
| Memory | $O(N)$ agent states | $O(M^2)$ matrix (sparse: $O(M \cdot K)$) |

### 8.4 When to use which method

**Use MC when:**
- You need individual-level histories (panel data, lifecycle paths)
- Your model doesn't yet have TM support (portfolio choice, health, habit)
- You want path-level statistics (percentiles, Gini, mobility)
- You're doing method of simulated moments (MSM) estimation

**Use TM when:**
- You need precise steady-state aggregates with no sampling noise
- You're computing SSJ Jacobians for HANK models
- You need impulse response functions to anticipated deviations (perfect-foresight transition paths; sometimes loosely called "MIT shocks," though true MIT shocks are unanticipated one-time surprises—see note below)
- Speed matters and you can use Harmenberg's trick (Section 11)

**Use both when:**
- **Cross-validation:** if MC and TM disagree, the TM grid is probably too coarse
- **Development workflow:** TM for quick steady-state checks, MC for distributions
- **Publication:** TM aggregates for precision, MC for individual-level moments

> **Terminology note: "MIT shocks."** The SSJ literature uses "MIT shock" to mean an unanticipated, one-time perturbation to a parameter (e.g., a surprise interest rate change at $t=0$), after which agents have perfect foresight about the transition path back to steady state. The SSJ Jacobian $\mathbf{J}^Y_Z$ characterizes the linearized response to such a shock. In contrast, some applied work uses "MIT shock" loosely to describe any anticipated deviation experiment. We prefer the precise term **"perfect-foresight transition path"** for the anticipated case and reserve **"MIT shock"** for the unanticipated-surprise interpretation.

## 9. Formal Convergence Properties

### 9.1 MC convergence (Santos and Peralta-Alva, 2005)

Under contraction:

$$
\left| \int h \, d\mu_a^* - \int h \, d\hat{\mu}_a^* \right| \leq \frac{L_h}{1 - \lambda} \left\| \Phi - \hat{\Phi} \right\|_\infty
$$

where $\Phi$ is the composite one-period transition, $\lambda < 1$ is the contraction rate, and $L_h$ is the Lipschitz constant of $h$.

### 9.2 TM convergence (Reiter, 2009)

The discretized $\boldsymbol{\Pi}$ converges to the exact operator as the grid is refined.

### 9.3 Joint convergence

Both methods converge to the true aggregate $\bar{h}^*$ from different directions:

$$
\underbrace{\hat{h}^{N,T}_{\text{MC}}}_{\text{noisy, unbiased}} \quad \xrightarrow[N,T \to \infty]{} \quad \bar{h}^* \quad \xleftarrow[M \to \infty]{} \quad \underbrace{\tilde{h}^M_{\text{TM}}}_{\text{deterministic, biased}}.
$$

If MC and TM disagree, the discrepancy decomposes:

$$
\hat{h}_{\text{MC}} - \tilde{h}_{\text{TM}} = \underbrace{(\hat{h}_{\text{MC}} - \bar{h}^*)}_{\text{MC sampling error}} + \underbrace{(\bar{h}^* - \tilde{h}_{\text{TM}})}_{\text{TM discretization bias}}.
$$

### 9.4 Contraction rate

The effective discount factor $\beta_{\text{period}} = \prod_{s \in \mathbb{S}} \beta_s$ bounds both value function convergence and the sensitivity of the invariant distribution to policy perturbations:

$$
\|\mathcal{V}_n - \mathcal{V}^*\| \leq \beta_{\text{period}}^n \|\mathcal{V}_0 - \mathcal{V}^*\|
$$

## 10. Extension: Markov-Switching Models

When agents face a discrete exogenous Markov state $j \in \{0, \ldots, J{-}1\}$ with transition matrix $\mathbf{M}$ ($M_{jj'} = \Pr(j' \mid j)$, row-stochastic), the one-period forward operator $\mathcal{T}^*$ acts on the joint distribution over $(m, j)$.

### 10.1 Block-structured transition matrix

The TM state space has $N = M \times J$ states, organized as $J$ blocks of $M$ grid points each. The $(M \times J) \times (M \times J)$ transition matrix $\boldsymbol{\Pi}$ has block structure:

$$
\boldsymbol{\Pi} = \begin{pmatrix}
\boldsymbol{\Pi}_{0 \to 0} & \boldsymbol{\Pi}_{1 \to 0} & \cdots \\
\boldsymbol{\Pi}_{0 \to 1} & \boldsymbol{\Pi}_{1 \to 1} & \cdots \\
\vdots & & \ddots
\end{pmatrix}
$$

where block $\boldsymbol{\Pi}_{j \to j'}$ ($M \times M$) captures transitions from Markov state $j$ to state $j'$. Each column is constructed by evaluating the state-$j$ policy, computing next-period resources under state-$j'$ parameters ($R_{j'}, \Gamma_{j'}$), applying the lottery method, and weighting by $M_{jj'} \cdot \text{LivPrb}_j$.

> **Timing convention.** In HARK, the interest rate $R_{j'}$ and permanent income growth factor $\Gamma_{j'}$ depend on the **target** (next-period) Markov state $j'$, not the source state $j$. This reflects the timing $m_{t+1} = R_{j'} \cdot a_t / (\Gamma_{j'} \psi_{t+1}) + \theta_{t+1}$, where $j'$ is the state that prevails when income is received. The consumption function $c^*_j(m)$ used to determine $a_t = m - c^*_j(m)$ depends on the **source** state $j$ (the state at the time of the decision).

### 10.2 Ergodic distribution

The ergodic distribution $\mathbf{p}^* \in \mathbb{R}^{M \times J}$ satisfies $\mathbf{p}^* = \boldsymbol{\Pi} \mathbf{p}^*$. Its marginal over the Markov state should match the analytical stationary distribution of $\mathbf{M}$—a useful validation check.

### 10.3 HARK implementation

`MarkovConsumerType.compute_pe_steady_state()` orchestrates the full pipeline. The numba-compiled `gen_tran_matrix_1D_markov()` constructs $\boldsymbol{\Pi}$ efficiently. Demonstrated in notebooks 01–02.

## 11. The Harmenberg Neutral Measure

### 11.1 The problem with permanent income

When $\Gamma \neq 1$ (or varies across Markov states), the distribution of permanent income $p$ is non-degenerate. The full state space becomes $(m, p, j)$ and the transition matrix grows as $(M_m \times M_p \times J)^2$—often intractable. Worse, the ergodic distribution of $p$ has a long right tail causing severe **p-grid truncation** errors in level aggregates.

### 11.2 The neutral measure

Harmenberg (2021) introduces a change of measure that eliminates the $p$ dimension. Define the **permanent-income-neutral measure** by reweighting the permanent shock probabilities:

$$
q^*(\psi_k) = \psi_k \cdot q(\psi_k), \qquad \text{so that } \mathbb{E}^*[1/\psi] = 1.
$$

Under this measure, the normalized transition $m' = R \cdot a / (\psi \cdot \Gamma) + \theta$ defines a valid Markov chain on $m$ alone, and the grid collapses from $(m, p)$ to just $m$.

### 11.3 Aggregation identity

$$
\bar{C}_{\text{level}} = \mathbb{E}^*[c(m)] \times \overline{p},
$$

where $\overline{p}$ is the mean permanent income level (computable analytically). Note: $\mathbb{E}^*[c(m)] \neq \mathbb{E}[c(m)]$ because the neutral-measure aggregate is $\mathbb{E}[c(m) \cdot p] / \mathbb{E}[p]$.

### 11.4 Critical pitfall: neutral measure is for the transition matrix only

> **Warning.** The neutral-measure income distribution must **only** be used when building the transition matrix $\boldsymbol{\Pi}$. The individual consumption-saving problem (the Bellman equation in Section 3.4) must **always** be solved using the true (standard) income shock distribution $Q(\psi, \theta)$.
>
> Concretely: `agent.solve()` must use the original shock probabilities $q(\psi_k)$, because the policy functions $c^*(m)$ and $v(m)$ are defined under the agent's actual expectations. Only the distribution-propagation step (`calc_transition_matrix`) uses the reweighted probabilities $q^*(\psi_k) = \psi_k \cdot q(\psi_k)$.
>
> Solving the model with neutral-measure shocks produces **incorrect policy functions**—a subtle bug that silently corrupts all downstream aggregates. The error is difficult to detect because the consumption function will still be smooth and monotone, but it will be quantitatively wrong.

### 11.5 HARK implementation

Set `agent.neutral_measure = True` before `update_income_process()`. HARK handles the separation automatically: the solver receives the true shock distribution while the TM builder receives the neutral-measure distribution. See notebooks 03–04 for the 2D-grid problem and its resolution via Harmenberg.

## 12. General Equilibrium: TM in Krusell–Smith

### 12.1 The aggregate state problem

In the Krusell–Smith (1998) framework, prices ($R$, $W$) are determined by aggregate capital $K$ through a production function. The individual consumption function becomes $c_j(m, M)$, where $M$ is the aggregate state. To build a 1D transition matrix, **fix** $M_t$ and evaluate $c_{j,t}(m) \equiv c_j(m, M_t)$.

This produces a time-varying sequence of transition matrices $\boldsymbol{\Pi}_0, \boldsymbol{\Pi}_1, \ldots$ that propagate the distribution forward deterministically.

### 12.2 Performance

In our experiments (notebook 08), TM propagation over 11,000 periods took ~2.5 seconds vs. ~243 seconds for MC with 5,000 agents—roughly 100$\times$ faster. Correlation between MC and TM trajectories exceeded 0.99.

In HARK: `CobbDouglasMarkovEconomy.make_history_tm()`.

## 13. Sequence-Space Jacobians

### 13.1 From TM to Jacobians

The transition matrix is the key input for computing **sequence-space Jacobians** (Auclert et al., 2021). The Jacobian $\mathbf{J}^Y_Z$ captures how the path of aggregate $Y$ responds to a one-time shock to parameter $Z$:

$$
(\mathbf{J}^Y_Z)_{ts} = \frac{\partial Y_t}{\partial Z_s}.
$$

### 13.2 The Fake News Algorithm

The Fake News Algorithm computes $\mathbf{J}$ using four ingredients derived from the steady-state TM: (1) direct effect on aggregates (curly $\mathcal{Y}$), (2) direct effect on distribution (curly $\mathcal{D}$), (3) expectation vectors, and (4) the Fake News matrix $\mathbf{F}$. For an $(M \times J)$-state Markov model, 50$\times$50 Jacobians were computed in ~0.3 seconds (notebook 09).

In HARK: `MarkovConsumerType.calc_jacobian(shk_param, T)`.

## 14. Further Extensions

### 14.1 Branching (mortality)

When a stage has branching (e.g., survival probability $p_{\text{live}}$), the transition matrix splits the probability mass:

$$
\boldsymbol{\Pi}_{\text{col}\ j} = p_{\text{live}} \cdot \text{lottery}(\text{transition from } g_j) + (1 - p_{\text{live}}) \cdot \mathbf{d}_{\text{newborn}}.
$$

### 14.2 The newborn distribution $\mathbf{d}_{\text{newborn}}$

When agents die (with probability $1 - p_{\text{live}}$ per period) and are immediately replaced, the probability mass of deceased agents is redistributed according to a **newborn distribution** $\mathbf{d}_{\text{newborn}} \in \mathbb{R}^{M}$ (or $\mathbb{R}^{M \times J}$ for Markov models). This distribution specifies where on the $(m, j)$ grid reborn agents begin.

Common choices include:

- **Point mass at initial assets.** All newborns start at a fixed $m_0$ (e.g., $m_0 = 1$). In the TM, this is a lottery assignment of the point $m_0$ onto the two nearest grid points. This is HARK's default behavior.

- **Markov-stationary initial state.** In Markov models, newborn agents' discrete state $j$ is drawn from the stationary distribution $\boldsymbol{\pi}^*$ of $\mathbf{M}$, so that $d_{\text{newborn},j} \propto \pi^*_j$.

- **Transitory shock lottery for $m$.** Newborns draw a transitory shock $\theta$ (but receive no permanent shock, i.e., $p = 1$), yielding initial market resources $m_0 = 1 + \theta$. The TM version integrates over the discretized $\theta$ distribution.

The choice of $\mathbf{d}_{\text{newborn}}$ affects the ergodic distribution $\mathbf{p}^*$, especially at high mortality rates. In HARK, `correct_newborn_dist` controls whether the TM uses a corrected newborn distribution that accounts for the transitory shock lottery, improving agreement between MC and TM steady states. When `neutral_measure = True`, the newborn distribution must also be expressed in neutral-measure units.

### 14.3 Multi-stage periods and post-decision shocks

The perch framework extends naturally to multi-stage periods (e.g., consumption then portfolio choice) and to post-decision shocks. These extensions are not exercised in the current notebook series but are straightforward generalizations.

## 15. Finite Horizon Extension

For a life-cycle model with $T$ periods, the policy functions are period-dependent: $\pi_t^*(x_v)$, $t = 0, 1, \ldots, T-1$.

**MC:** Each agent draws shocks and traverses the perch structure at each age, following age-dependent policies.

**TM:** There is a sequence of transition matrices $\boldsymbol{\Pi}_0, \boldsymbol{\Pi}_1, \ldots, \boldsymbol{\Pi}_{T-1}$:

$$
\mathbf{p}_{a,t+1} = \boldsymbol{\Pi}_t \, \mathbf{p}_{a,t}, \qquad t = 0, 1, \ldots, T-1.
$$

There is no ergodic distribution; the distribution at each age is transient.

## 16. Summary of Mathematical Objects

| Object | Exact | MC approximation | TM approximation |
|--------|-------|-------------------|-------------------|
| Arrival space $\mathcal{X}_a$ | Continuous | Continuous (agents live in $\mathcal{X}_a$) | Finite grid $\mathcal{G}_a \subset \mathcal{X}_a$ |
| Arrival measure $\mu_{a,t}$ | Probability measure | Empirical $\hat{\mu}^N_{a,t} = \frac{1}{N}\sum_i \delta_{x_a^{(i)}}$ | Probability vector $\mathbf{p}_{a,t} \in \mathbb{R}^M$ |
| Transition $\Gamma_{av}$ | Integral over shock dist. | $N$ independent shock draws | Discrete sum over $K$ quadrature points |
| Transition $\Gamma_{ve}$ | Pushforward by $\pi^*$ | Evaluate $\pi^*$ at each agent's $x_v$ | Evaluate $\pi^*$ at each grid point |
| Connector $\Gamma_{ea+}$ | Pushforward by $g_{ea+}$ | Exact mapping per agent | Lottery projection onto grid |
| Composite $\mathcal{T}^*$ | $\Gamma_{ea+} \circ \Gamma_{ve} \circ \Gamma_{av}$ | Per-agent sequential traversal | Matrix multiply $\boldsymbol{\Pi}$ |
| Ergodic dist. $\mu_a^*$ | Fixed point of $\mathcal{T}^*$ | Long-run empirical dist. | Eigenvector of $\boldsymbol{\Pi}$ |
| Aggregate $\bar{h}$ | $\int h \, d\mu_v$ | Sample mean $\frac{1}{N}\sum h(x_v^{(i)})$ | Dot product $\mathbf{h}^\top \mathbf{p}_v$ |
| Error type | — | Variance $O(1/N)$ (at $\Gamma_{av}$) | Bias $O(\Delta g)$ (at $\Gamma_{ea+}$) |

## 17. Notebook Guide

The following notebooks in `sims-about/` progressively demonstrate the concepts in this framework.

| # | Notebook | Model | Framework sections |
|---|----------|-------|-----------|
| 1 | `01-markov-tm-prototype` | 2-state Markov, $\Gamma=1$ | Secs 7, 10 (1D TM, block structure) |
| 2 | `02-serial-unemployment-tm` | 4-state serial unemployment | Sec 10 (scaling to more states) |
| 3 | `03-serial-growth-tm-2d` | 5-state, $\Gamma \neq 1$ | Sec 11.1 (2D grid problem) |
| 4 | `04-serial-growth-tm-harmenberg` | 5-state, Harmenberg | Sec 11 (neutral measure) |
| 5 | `05-tm-consolidation` | Single-state, validation | Secs 7–8 (TM vs NK built-in) |
| 6 | `06-agg-shock-markov-tm` | Krusell–Smith economy | Sec 12 (2D cFunc, fixed $M$) |
| 7 | `07-validate-markov-tm-methods` | 2-state, production code | Sec 10 (validate `MarkovConsumerType`) |
| 8 | `08-tm-in-ks` | Krusell–Smith, TM propagation | Sec 12 (TM-in-KS loop) |
| 9 | `09-markov-ssj` | 2-state, Jacobians | Sec 13 (SSJ via Fake News) |

See also Will Du's `examples/SequenceSpaceJacobians/Transition_Matrix_Example.ipynb` for the original MC vs TM comparison on the single-state IndShock model (Sections 6–9).

## References

- Achdou, Y., Han, J., Lasry, J.-M., Lions, P.-L., and Moll, B. (2022). Income and Wealth Distribution in Macroeconomics: A Continuous-Time Approach. *Review of Economic Studies*, 89(1), 45–86.
- Algan, Y., Allais, O., den Haan, W. J., and Rendahl, P. (2014). Solving and Simulating Models with Heterogeneous Agents and Aggregate Uncertainty. In *Handbook of Computational Economics*, Vol. 3, pp. 475–529. Elsevier.
- Auclert, A., Bardóczy, B., Rognlie, M., and Straub, L. (2021). Using the Sequence-Space Jacobian to Solve and Estimate Heterogeneous-Agent Models. *Econometrica*, 89(5), 2375–2408.
- den Haan, W. J. (2010). Comparison of Solutions to the Incomplete Markets Model with Aggregate Uncertainty. *Journal of Economic Dynamics and Control*, 34(1), 4–27.
- Harmenberg, K. (2021). Aggregating Heterogeneous-Agent Models with Permanent Income Shocks. *Journal of Economic Dynamics and Control*, 129, 104185.
- Krusell, P. and Smith, A. A. (1998). Income and Wealth Heterogeneity in the Macroeconomy. *Journal of Political Economy*, 106(5), 867–896.
- Reiter, M. (2009). Solving Heterogeneous-Agent Models by Projection and Perturbation. *Journal of Economic Dynamics and Control*, 33(3), 649–665.
- Santos, M. S. and Peralta-Alva, A. (2005). Accuracy of Simulations for Stochastic Dynamic Models. *Econometrica*, 73(6), 1939–1976.
- Young, E. R. (2010). Solving the Incomplete Markets Model with Aggregate Uncertainty Using the Krusell–Smith Algorithm and Non-Stochastic Simulations. *Journal of Economic Dynamics and Control*, 34(1), 36–41.

See `bibliography.md` in this directory for a comprehensive annotated bibliography with reading paths organized by topic.